# RAG chain over SEC 10-K filings

Single-turn RAG using pgvector for retrieval and GPT-4o-mini for generation.
Retrieval is filtered by `stock_symbol` so answers are grounded in a specific company's filing.

**Prerequisites:** Docker container `pgvector-rag` must be running (`docker start pgvector-rag`).

In [ ]:
import os

from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_postgres import PGVector

load_dotenv()

POSTGRES_URL = os.environ["POSTGRES_URL"]
OPENAI_KEY = os.environ["OPENAI_KEY"]
COLLECTION_NAME = "sec_10k_chunks"

## Connect to the vector store

In [ ]:
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=OPENAI_KEY,
)

store = PGVector(
    connection=POSTGRES_URL,
    embeddings=embeddings_model,
    collection_name=COLLECTION_NAME,
    use_jsonb=True,
)

print("Connected to vector store")

## Build the RAG chain

The chain:
1. Retrieves the top-k chunks from pgvector, filtered to a specific ticker
2. Formats them into a context string
3. Passes context + question to the LLM

In [ ]:
PROMPT_TEMPLATE = """\
You are a financial analyst assistant. Answer the question using only the provided excerpts \
from the company's 10-K annual filing. If the answer is not in the excerpts, say so.

Excerpts:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(PROMPT_TEMPLATE)
llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_KEY)


def format_docs(docs):
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('section', '')} | {doc.metadata.get('doc_id', '')}]\n{doc.page_content}"
        for doc in docs
    )


def make_rag_chain(ticker: str, k: int = 5, section: str = None):
    """Build a RAG chain filtered to a specific ticker (and optionally a section)."""
    filter_ = {"stock_symbol": ticker}
    if section:
        filter_["section"] = section

    retriever = store.as_retriever(
        search_kwargs={"k": k, "filter": filter_}
    )

    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )


print("RAG chain builder ready")

## Ask questions

Set `ticker` to any ingested symbol (e.g. `aapl`, `msft`, `avgo`) and ask away.

In [ ]:
chain = make_rag_chain(ticker="aapl")

answer = chain.invoke("What are Apple's main products and services?")
print(answer)

In [ ]:
# Restrict retrieval to Item 7 (MD&A) for financial questions
chain_mda = make_rag_chain(ticker="aapl", section="Item 7.")

answer = chain_mda.invoke("What drove Apple's revenue growth this year?")
print(answer)

In [ ]:
# Inspect retrieved chunks to understand what the LLM saw
question = "What are the main risk factors?"
ticker = "aapl"

retriever = store.as_retriever(
    search_kwargs={"k": 5, "filter": {"stock_symbol": ticker}}
)
docs = retriever.invoke(question)

for i, doc in enumerate(docs, 1):
    print(f"--- Chunk {i} | {doc.metadata.get('section')} | {doc.metadata.get('doc_id')} ---")
    print(doc.page_content[:400])
    print()